# tiny-yolov3-pets on pets — inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/EmbeddedComputervision/blob/main/notebooks/pets/tiny-yolov3-pets_infer.ipynb)

Role: inference; tier **R** on the first pass (`SUBSET = 50`), **N** on the full split (3662 images, `SUBSET = 0`). Tasks: [T32.6](../../docs/tasks/M32-trashnet-pets.md), [T32.12](../../docs/tasks/M32-trashnet-pets.md), [T14.6](../../docs/tasks/M14-notebooks.md), [T14.11](../../docs/tasks/M14-notebooks.md);
notebooks: [M14](../../docs/tasks/M14-notebooks.md).

Prerequisites:
- data: `data/pets/annotations_val.json` — tools/get_datasets.sh pets
- weights: `build/notebooks/pets/tiny-yolov3-pets/[runs/<run>/]final.weights` — notebooks/pets/tiny-yolov3-pets_train.ipynb or _sweep.ipynb
- cfg: `build/m11/cfg/tiny-yolov3-pets.cfg` — notebooks/pets/tiny-yolov3-pets_train.ipynb (preparation)

Each step prints the `python tools/…` command it runs: a result can be reproduced
from the command line. Outputs in `build/notebooks/pets/tiny-yolov3-pets/eval/`, never in `results/` (an mAP on `SUBSET`
images is not publishable, M12 rules). Notebook generated by
`python -m tools.notebooks` (`make notebooks`): edit the template
`tools/notebooks/gabarits.py`, not this file.

## Parameters

In [ ]:
NET          = 'build/m11/cfg/tiny-yolov3-pets.cfg'  # name in CFG_FILES or cfg (relative to the root)
DATASET      = 'pets'  # key of DATASETS
SPLIT        = None  # None: evaluation split of the dataset
WEIGHTS      = None  # None: final.weights of run RUN; or a path
DEVICE       = 'cpu'  # cpu | gpu (training only; the rest on CPU)
SUBSET       = 50  # images evaluated; 0: full split (tier N)
SIZE         = None  # input WxH (e.g. '640x192'); None: the one of the cfg
RESIZE       = 'stretch'  # stretch (M12 reference) | letterbox
METRIC       = None  # None: the one of the dataset (coco for coco and flir)
DATA_ROOT    = None  # dataset root; None: data/<dataset> (Colab: Drive folder)
RUNS_DIR     = 'build/notebooks/pets/tiny-yolov3-pets'  # runs of the _train and _sweep notebooks
RUN          = None  # None: the most recent; e.g. 'train', 'b16-sall', 'b8-s500'
COMPARE      = False  # evaluates every run and compares them (T14.11)
INT8         = False  # integer part: calibration and INT8 mAP (T14.4)
CALIB_IMAGES = 500  # calibration images (calib split of the dataset)
ACT_HIST     = False  # activation histograms L00-L04 (T11.3)
HW           = False  # hardware part (needs INT8): export, C++ golden, C-sim on SUBSET images, mAP at the three stages (T8.2, T24.5)
DRIVE_DIR    = '/content/drive/MyDrive/EmbeddedCV'  # Colab: archives data/<dataset>.tar; None: no Drive
JOBS         = None  # evaluation processes; None: one per CPU core, capped by the memory (Colab TPU runtime: dozens)
SHOW         = 6  # images shown
OUT          = 'build/notebooks/pets/tiny-yolov3-pets/eval'  # notebook outputs
REPO_URL     = 'https://github.com/lrivals/EmbeddedComputervision.git'  # Colab: cloned repository
REV          = 'main'  # Colab: revision

## Environment

Local: nothing is installed or downloaded, the cell checks the prerequisites and
stops with the command to run. Colab: clone of the repository, installation, CuPy if
`DEVICE = "gpu"`, weights, then data by `colab.prepare_data` (archive
`<DRIVE_DIR>/data/<dataset>.tar` or direct download; ExDark and FLIR: archive only,
made on the PC by `tools/get_datasets.sh pack <dataset>`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COLAB = "google.colab" in sys.modules


def repo_root():
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "tools" / "notebooks" / "commandes.py").exists():
            return d
    return None


ROOT = repo_root()
if COLAB:
    if ROOT is None:
        ROOT = Path("/content/EmbeddedComputervision")
        if not ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    # Clone of an earlier session: reset to REV (notebook and code of the same commit).
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "FETCH_HEAD"], check=True)
    for m in [m for m in sys.modules if m.split(".")[0] in ("tools", "yolo")]:
        del sys.modules[m]  # modules imported before the update
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}/python[data,plots]"], check=True)
    if DEVICE == "gpu":
        # CPU runtime: no driver, CuPy would say "cudaErrorInsufficientDriver".
        try:
            smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        except FileNotFoundError:
            smi = None
        if smi is None or smi.returncode != 0:
            raise RuntimeError("Colab runtime without a GPU: create a new Colab server "
                               "of GPU type (T4, L4 or A100), or DEVICE = 'cpu'")
        cuda = smi.stdout.split("CUDA Version:")[-1].split()[0]  # driver version
        cupy_pkg = "cupy-cuda13x" if int(cuda.split(".")[0]) >= 13 else "cupy-cuda12x"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", cupy_pkg],
                       check=True)
elif ROOT is None:
    raise RuntimeError("repository root not found: open the notebook from notebooks/")
os.chdir(ROOT)
for p in (ROOT, ROOT / "python"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tools.notebooks import commandes as C
from tools.notebooks import env

TRACE = env.trace(DEVICE)
env.check_device(DEVICE)
from tools.notebooks import colab, runs

if COLAB and DRIVE_DIR:  # runs trained on Colab: copied to <DRIVE_DIR>/runs/
    env.mount_drive(DRIVE_DIR)
    colab.restore_outputs(RUNS_DIR, DRIVE_DIR)
RUNS = runs.find_runs(RUNS_DIR)
print(f"runs of {RUNS_DIR}:\n{runs.listing(RUNS)}")
CHOSEN = None
if WEIGHTS is None:
    CHOSEN = runs.pick(RUNS_DIR, RUN, hint='notebooks/pets/tiny-yolov3-pets_train.ipynb or _sweep.ipynb')
    WEIGHTS, OUT = CHOSEN.weights, f"{OUT}/{CHOSEN.name}"
    print(f"chosen run: {CHOSEN.name} ({WEIGHTS})")
env.prepare(DATASET, WEIGHTS, NET, DATA_ROOT, COLAB, hint='notebooks/pets/tiny-yolov3-pets_train.ipynb or _sweep.ipynb',
            drive_dir=DRIVE_DIR)
Path(OUT).mkdir(parents=True, exist_ok=True)

## Network and weights

`yolo.models.tiny_yolo.build` and `yolo.io.darknet_weights.load_darknet_weights`.

In [ ]:
import json

import numpy as np
from IPython.display import Markdown, display
from PIL import Image

from tools.detect import draw
from tools.notebooks.matrice import SPLIT_IMAGES, palier
from yolo.data import datasets
from yolo.data.letterbox import input_size, parse_size, size_label
from yolo.infer.pipeline import detect, preprocess
from yolo.io.darknet_weights import load_darknet_weights
from yolo.models.tiny_yolo import build

net = build(NET)
load_darknet_weights(net, WEIGHTS)
size = input_size(net.net, parse_size(SIZE) if SIZE else None)
channels = net.net["input"][0]
family, names = datasets.model_family(DATASET, net.net["classes"])
print(f"{NET}: {net.net['classes']} classes ({family}), input {size_label(size)}, "
      f"{channels} channel(s); weights {WEIGHTS}")

## Detections on a few images

`yolo.infer.pipeline.preprocess` and `detect` (demo threshold), boxes by
`tools/detect.py:draw`.

In [ ]:
for s in datasets.load(DATASET, DATA_ROOT, SPLIT)[:SHOW]:
    with Image.open(s["image"]) as img:
        x, wh = preprocess(img, size, RESIZE, "pil", channels)
        boxes, scores, labels = detect(net, x[None], [wh], RESIZE)[0]
        shown = draw(img, boxes, scores, labels, names)
    shown.thumbnail((640, 640))
    print(f"{s['id']}: {len(labels)} detections")
    display(shown)

## Float mAP (T14.3)

`tools/eval_voc.py`, metric of the dataset (`Dataset.metric`; COCO and FLIR: AP@[.5:.95]
and AP50). PR curves of the automatic mode of M13 (VOC metric).

The chosen run is evaluated in `RUNS_DIR/eval/<run>/`, as by the comparison
below, which reuses this result: an mAP already computed for this `SUBSET` is not
computed again.

In [ ]:
print(f"{SUBSET or 'all'} images: tier "
      f"{palier(SUBSET or SPLIT_IMAGES.get(DATASET))} (M12 rules)")
if CHOSEN is not None:
    EVAL_DIR = Path(RUNS_DIR, "eval", CHOSEN.name)
    runs.evaluate(CHOSEN, NET, DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE,
                  DATA_ROOT, out_dir=EVAL_DIR, jobs=JOBS)
    map_float = EVAL_DIR / f"map_float_s{SUBSET or 'all'}.md"
    dets = EVAL_DIR / f"dets_s{SUBSET or 'all'}"
else:
    map_float = Path(OUT) / "map_float.md"
    map_float.unlink(missing_ok=True)
    C.run(C.cmd_eval_voc(NET, WEIGHTS, DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE,
                         DATA_ROOT, out=f"{OUT}/dets", markdown=map_float, jobs=JOBS))
    dets = Path(OUT, "dets")
display(Markdown(map_float.read_text()))
for png in sorted(Path(dets, "figures").glob("pr*.png")):
    display(Image.open(png))

## Comparison of the runs (T14.11)

With `COMPARE = True`, each run found in `RUNS_DIR` (`_train` notebook and
`_sweep` sweep) is evaluated on the same `SUBSET` images; an mAP already computed
for this `SUBSET` is reused. The run evaluated above is chosen by `RUN`.

In [ ]:
if COMPARE:
    rows = [runs.evaluate(r, NET, DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE,
                          DATA_ROOT, out_dir=f"{RUNS_DIR}/eval/{r.name}", jobs=JOBS)
            for r in RUNS]
    display(Markdown(runs.table(rows)))
else:
    print("COMPARE = False: comparison skipped; runs found:")
    print(runs.listing(RUNS))

## Integer part (T14.4)

Enabled by `INT8 = True`. Calibration by `tools/calibrate.py` on the `calib` split
of the dataset (scales and report in `OUT`, not in `results/`), then
`tools/eval_quant.py --variants float,int`. On the full split (`SUBSET = 0`),
tier **N**: count 12 to 25 min for VOC2007 test on top of the calibration.

In [ ]:
map_int = None
if INT8:
    if not SUBSET:
        print("SUBSET = 0: full split, tier N (several tens of minutes)")
    calib = Path(OUT) / "calib.json"
    C.run(C.cmd_calibrate(NET, WEIGHTS, DATASET, calib, Path(OUT) / "calibration.md",
                          CALIB_IMAGES, data_root=DATA_ROOT))
    map_int = Path(OUT) / f"map_{RESIZE}_{SUBSET or 'complet'}.json"
    HW_DIR = Path(OUT) / "hw"  # integer detections per image, for the hardware part
    HW_DIR.mkdir(parents=True, exist_ok=True)
    C.run(C.cmd_eval_quant(NET, map_int, "float,int", SUBSET, JOBS, DATASET,
                           weights=WEIGHTS, calib=calib, metric=METRIC, split=SPLIT,
                           size=SIZE, data_root=DATA_ROOT,
                           save_dets=HW_DIR / "int.jsonl" if HW else None))
    d = json.loads(map_int.read_text())
    fl, it = d["results"]["float"], d["results"]["int"]
    rows = ["| class | float | integer | gap |", "|---|---|---|---|"]
    rows += [f"| {c} | {100 * a:.2f} | {100 * b:.2f} | {100 * (b - a):+.2f} |"
             for c, a, b in zip(d["classes"], fl["aps"], it["aps"])]
    rows.append(f"| **mAP** | **{100 * fl['map']:.2f}** | **{100 * it['map']:.2f}** | "
                f"{100 * (it['map'] - fl['map']):+.2f} |")
    display(Markdown(f"{d['images']} images, {d['metric']}\n\n" + "\n".join(rows)))
    for png in sorted((map_int.parent / "figures").glob(f"*{map_int.stem.removeprefix('map_')}*.png")):
        display(Image.open(png))
    if ACT_HIST:
        others = "voc" if DATASET == "voc" else f"voc,{DATASET}"
        C.run(C.cmd_act_hist(NET, WEIGHTS, others, calib, Path(OUT) / "act_hist"))
        display(Image.open(Path(OUT) / "act_hist" / "act_hist.png"))
else:
    print("INT8 = False: integer part skipped")

## Hardware part (T8.2, T24.5)

Enabled by `HW = True` (with `INT8 = True`: same calibration, integer detections of
the previous cell). Chain of the `coco-v4` profile of `tools/m11.sh`:

1. `tools/export_model.py`: integer model (manifest, blobs) and dumps of 3 images of
   the dataset in `OUT/model`;
2. C++ golden (`golden_run`) == Python dumps to the byte, layer by layer and
   detections (`tools/compare_dumps.py`, as `make golden-check`);
3. FPGA stage in C-sim: the HLS kernel compiled without `ap_int` behind the ARM
   driver (`yolo_bench`, `tools/bench_sim.sh`), on the int8 inputs of the integer
   model (`tools/make_inputs.py`), for the first `SUBSET` images;
4. `tools/map_stades.py`: mAP at the three stages and integer == C-sim equality image
   by image; `tools/perf_model.py`: cycles and ms of the current kernel (projection).

Count ≈ 1 s of C-sim per image and per core (C-sim interrupted: run the cell again,
the finished batches are kept as long as the exported model does not change).

In [ ]:
import hashlib
import shutil

map_hw = None
if HW:
    if not INT8:
        raise ValueError("HW = True needs INT8 = True (calibration and integer "
                         "detections)")
    model = Path(OUT) / "model"
    C.run(C.cmd_export(NET, WEIGHTS, calib, model, DATASET, DATA_ROOT))
    for c in C.cmd_cmake("cpp/golden", "build/golden", "golden_run"):
        C.run(c)
    for dump in sorted(p for p in (model / "dumps").iterdir() if p.is_dir()):
        got = HW_DIR / "golden" / dump.name
        C.run(C.cmd_golden_run(model, dump / "input.npy", got))
        C.run(C.cmd_compare_dumps(dump, got))
    # C-sim detections of another exported model: erased (otherwise resumed).
    blobs = hashlib.sha256(b"".join((model / f).read_bytes() for f in
                                    ("weights.bin", "bias.bin", "requant.bin")))
    stamp = HW_DIR / "model.sha256"
    if not stamp.exists() or stamp.read_text() != blobs.hexdigest():
        shutil.rmtree(HW_DIR / "sim", ignore_errors=True)
        stamp.write_text(blobs.hexdigest())
    C.run(C.cmd_make_inputs(NET, DATASET, SUBSET, HW_DIR, DATA_ROOT))
    shutil.copy(map_int, HW_DIR / "eval_float_int.json")
    for c in C.cmd_cmake("sw", "build/sw-fast", "yolo_bench",
                         "-DACC_NO_APINT -march=native", ["-DSW_BACKEND=sim"]):
        C.run(c)
    C.run(C.cmd_bench_sim(HW_DIR, model, JOBS), env={"BENCH": C.BENCH_FAST})
    map_hw = HW_DIR / "map_stades.md"
    C.run(C.cmd_map_stades(NET, DATASET, HW_DIR, map_hw, SUBSET, DATA_ROOT))
    display(Markdown(map_hw.read_text()))
    C.run(C.cmd_perf_model(model / "manifest.json"))
else:
    print("HW = False: hardware part skipped")

## Summary

In [ ]:
print(map_float.read_text())
if map_int:
    d = json.loads(map_int.read_text())
    for v, r in d["results"].items():
        print(f"{v:6s} mAP {100 * r['map']:.2f} ({d['images']} images)")
if map_hw:
    print(map_hw.read_text().split("## AP")[0])
print(f"outputs: {OUT}/; revision {TRACE['rev']}")
print("equivalent commands:")
for c in C.HISTORY:
    print("  " + c)